## IMPORTS

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import sys
import duckdb
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Import shared utilities
sys.path.insert(0, os.path.abspath('.'))
from notebook_utils import (
    drop_high_missing_columns, impute_features,
    plot_trajectory_distribution, plot_boxplots_with_stats,
    plot_roc_pr_curves, print_statistical_comparisons, train_repeated_cv, biomarker_summary_stats
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ Imports successful")
print("✓ Shared utilities loaded")

✓ Imports successful
✓ Shared utilities loaded


## LOAD DATA

In [2]:
# Connect to HiRiD DuckDB
db_path = '/home/gaga/data/physionet/HiRiD/hirid.duckdb'
conn = duckdb.connect(db_path, read_only=True)

print(f"✓ Connected to HiRiD database: {db_path}")

✓ Connected to HiRiD database: /home/gaga/data/physionet/HiRiD/hirid.duckdb


In [3]:
# Load mechanically ventilated patients
# Define as patients with ventilator mode or PEEP > 0

n_sample_patients = 20000

# Step 1: Get patient IDs on mechanical ventilation
ventilated_query = f"""
SELECT DISTINCT CAST(o.patientid AS INTEGER) as patientid
FROM observations o
WHERE 
    (
        -- Ventilator mode
        o.variableid = '3845'
        -- PEEP > 0
        OR (o.variableid IN ('2600', '2610') AND CAST(o.value AS DOUBLE) > 0)
        -- Tidal volume
        OR (o.variableid IN ('2410', '2400') AND CAST(o.value AS DOUBLE) > 0)
    )
LIMIT {n_sample_patients}
"""

ventilated_patients = conn.execute(ventilated_query).fetchdf()['patientid'].tolist()
print(f"Step 1: Found {len(ventilated_patients):,} mechanically ventilated patients")

# Step 2: Get patient demographics
patient_query = f"""
SELECT 
    CAST(g.patientid AS INTEGER) as patientid,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time,
    g.sex,
    CAST(g.age AS INTEGER) as age,
    g.discharge_status,
    COUNT(DISTINCT o.datetime) as n_observations,
    EPOCH(MAX(CAST(o.datetime AS TIMESTAMP)) - MIN(CAST(o.datetime AS TIMESTAMP))) / 86400.0 as los_days
FROM ref_general_table g
INNER JOIN observations o ON g.patientid = o.patientid
WHERE CAST(g.patientid AS INTEGER) IN {tuple(ventilated_patients)}
GROUP BY g.patientid, g.admissiontime, g.sex, g.age, g.discharge_status
HAVING 
    EPOCH(MAX(CAST(o.datetime AS TIMESTAMP)) - MIN(CAST(o.datetime AS TIMESTAMP))) / 86400.0 >= 2.0
    AND COUNT(DISTINCT o.datetime) >= 100
"""

patient_df = conn.execute(patient_query).fetchdf()

print(f"\n✓ Loaded {len(patient_df):,} ventilated patients")
print(f"  Mean LOS: {patient_df['los_days'].mean():.1f} days")
print(f"  Mean age: {patient_df['age'].mean():.1f} years")

Step 1: Found 20,000 mechanically ventilated patients

✓ Loaded 6,892 ventilated patients
  Mean LOS: 6.7 days
  Mean age: 62.6 years


In [4]:
# Sample subset for analysis
n_patients = min(5000, len(patient_df))
patient_subset = patient_df.sample(n_patients, random_state=920)['patientid'].tolist()

print(f"✓ Using {len(patient_subset):,} patients for analysis")

✓ Using 5,000 patients for analysis


In [5]:
# Load PaO2 (arterial oxygen partial pressure)
pao2_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as pao2,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '20000200'
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) > 0
    AND CAST(o.value AS DOUBLE) <= 700  -- Remove outliers (mmHg)
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

pao2_df = conn.execute(pao2_query).fetchdf()
print(f"✓ Loaded {len(pao2_df):,} PaO2 measurements")

# Load FiO2 (fraction of inspired oxygen)
fio2_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(o.value AS DOUBLE) as fio2,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '2010'
    AND o.value IS NOT NULL
    AND CAST(o.value AS DOUBLE) >= 21
    AND CAST(o.value AS DOUBLE) <= 100  -- FiO2 as percentage
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

fio2_df = conn.execute(fio2_query).fetchdf()
print(f"✓ Loaded {len(fio2_df):,} FiO2 measurements")

# Merge PaO2 and FiO2 within 1-hour window and calculate P/F ratio
pao2_df['charttime'] = pd.to_datetime(pao2_df['charttime'])
fio2_df['charttime'] = pd.to_datetime(fio2_df['charttime'])

pf_df = pd.merge_asof(
    pao2_df.sort_values('charttime'),
    fio2_df.sort_values('charttime')[['patientid', 'charttime', 'fio2']],
    on='charttime',
    by='patientid',
    tolerance=pd.Timedelta(hours=1),
    direction='nearest'
)

# Calculate P/F ratio
pf_df = pf_df.dropna(subset=['fio2'])
pf_df['pf_ratio'] = (pf_df['pao2'] / pf_df['fio2']) * 100

print(f"\n✓ Calculated P/F ratio for {len(pf_df):,} time points")
print(f"  Patients with P/F ratio: {pf_df['patientid'].nunique():,}")
if len(pf_df) > 0:
    print(f"  Mean P/F ratio: {pf_df['pf_ratio'].mean():.1f}")
    print(f"  Median P/F ratio: {pf_df['pf_ratio'].median():.1f}")

✓ Loaded 113,794 PaO2 measurements
✓ Loaded 12,782,311 FiO2 measurements

✓ Calculated P/F ratio for 88,870 time points
  Patients with P/F ratio: 4,874
  Mean P/F ratio: 217.8
  Median P/F ratio: 202.3


In [6]:
# Load ventilator mode to identify ventilation periods
mode_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    o.value as vent_mode,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid = '3845'
    AND o.value IS NOT NULL
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading ventilator mode data...")
mode_df = conn.execute(mode_query).fetchdf()

print(f"\n✓ Loaded {len(mode_df):,} ventilator mode records")
print(f"  Patients with mode data: {mode_df['patientid'].nunique():,}")
if len(mode_df) > 0:
    print(f"  Unique modes: {mode_df['vent_mode'].nunique()}")

Loading ventilator mode data...

✓ Loaded 3,374,717 ventilator mode records
  Patients with mode data: 3,899
  Unique modes: 13


In [7]:
# Load SpO2 and respiratory rate for outcome definition
resp_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    o.variableid,
    CAST(o.value AS DOUBLE) as value,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
WHERE 
    o.variableid IN (
        '4000', '8280',  -- SpO2
        '300', '310', '5685'  -- Respiratory rate
    )
    AND o.value IS NOT NULL
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading respiratory parameters...")
resp_df = conn.execute(resp_query).fetchdf()

print(f"\n✓ Loaded {len(resp_df):,} respiratory parameter measurements")

Loading respiratory parameters...

✓ Loaded 73,104,438 respiratory parameter measurements


In [ ]:
# Load comprehensive vitals and labs using variable reference table
vitals_labs_query = f"""
SELECT 
    CAST(o.patientid AS INTEGER) as patientid,
    CAST(o.datetime AS TIMESTAMP) as charttime,
    CAST(g.admissiontime AS TIMESTAMP) as admission_time,
    o.variableid,
    v.variablename,
    CAST(o.value AS DOUBLE) as value
FROM observations o
INNER JOIN ref_general_table g ON o.patientid = g.patientid
INNER JOIN ref_hirid_variable_reference v ON o.variableid = v.variableid
WHERE 
    o.variableid IN (
        '200',   -- Heart rate
        '210',   -- Temperature
        '300', '310', '5685',  -- Respiratory rate
        '400', '410', '600', '610', '620',  -- BP (invasive)
        '3000', '3010',  -- SpO2
        '24000100', '24000200', '24000300',  -- Glucose
        '24000500', '24000510', '24000520',  -- Sodium
        '24000600', '24000610', '24000620',  -- Potassium
        '24000700',  -- Chloride
        '24000800',  -- Bicarbonate
        '24001000',  -- Calcium
        '24001100',  -- Magnesium
        '24001200',  -- Phosphate
        '24001300', '24001310',  -- BUN
        '24001400', '24001410',  -- Creatinine
        '24001500', '24001510',  -- Hemoglobin
        '24001600',  -- WBC
        '24001700',  -- Platelets
        '24001900',  -- INR
        '24002000',  -- Albumin
        '24002100',  -- Lactate
        '15000500'   -- Weight
    )
    AND o.value IS NOT NULL
    AND CAST(o.patientid AS INTEGER) IN {tuple(patient_subset)}
ORDER BY o.patientid, o.datetime
"""

print("Loading comprehensive vitals and labs...")
vitals_labs_df = conn.execute(vitals_labs_query).fetchdf()

print(f"\n✓ Loaded {len(vitals_labs_df):,} vitals/labs measurements")
print(f"  Patients: {vitals_labs_df['patientid'].nunique():,}")
print(f"  Unique variables: {vitals_labs_df['variableid'].nunique()}")

In [ ]:
# Process vitals/labs: map variable IDs to column names
vitals_labs_df['charttime'] = pd.to_datetime(vitals_labs_df['charttime'])
vitals_labs_df['admission_time'] = pd.to_datetime(vitals_labs_df['admission_time'])
vitals_labs_df['time_days'] = (vitals_labs_df['charttime'] - vitals_labs_df['admission_time']).dt.total_seconds() / 86400
vitals_labs_df['time_day'] = vitals_labs_df['time_days'].astype(int)

# Map variable IDs to readable names
var_mapping = {
    '200': 'heart_rate',
    '210': 'temperature',
    '300': 'respiratory_rate', '310': 'respiratory_rate', '5685': 'respiratory_rate',
    '400': 'sbp', '410': 'sbp', '600': 'dbp', '610': 'dbp', '620': 'map',
    '3000': 'spo2', '3010': 'spo2',
    '24000100': 'glucose', '24000200': 'glucose', '24000300': 'glucose',
    '24000500': 'sodium', '24000510': 'sodium', '24000520': 'sodium',
    '24000600': 'potassium', '24000610': 'potassium', '24000620': 'potassium',
    '24000700': 'chloride',
    '24000800': 'bicarbonate',
    '24001000': 'calcium',
    '24001100': 'magnesium',
    '24001200': 'phosphate',
    '24001300': 'bun', '24001310': 'bun',
    '24001400': 'creatinine', '24001410': 'creatinine',
    '24001500': 'hemoglobin', '24001510': 'hemoglobin',
    '24001600': 'wbc',
    '24001700': 'platelets',
    '24001900': 'inr',
    '24002000': 'albumin',
    '24002100': 'lactate',
    '15000500': 'weight'
}

vitals_labs_df['variable_name'] = vitals_labs_df['variableid'].map(var_mapping)

# Unit conversions
vitals_labs_df.loc[vitals_labs_df['variable_name'] == 'creatinine', 'value'] /= 88.4  # µmol/L to mg/dL

# Apply clinically plausible ranges
range_filters = {
    'heart_rate': (20, 250),
    'temperature': (30, 45),
    'respiratory_rate': (5, 60),
    'sbp': (40, 250),
    'dbp': (20, 180),
    'map': (30, 200),
    'spo2': (50, 100),
    'glucose': (20, 800),
    'sodium': (100, 180),
    'potassium': (1.5, 10),
    'creatinine': (0.1, 20),
    'hemoglobin': (3, 20),
    'wbc': (0.1, 100),
    'platelets': (1, 2000)
}

for var, (low, high) in range_filters.items():
    mask = vitals_labs_df['variable_name'] == var
    vitals_labs_df.loc[mask, 'value'] = vitals_labs_df.loc[mask, 'value'].clip(low, high)

print(f"\n✓ Processed vitals/labs with variable mapping and range validation")
print(f"  Unique variables: {vitals_labs_df['variable_name'].nunique()}")

In [ ]:
# Aggregate vitals/labs by time_day: min, max, mean
vitals_labs_agg = vitals_labs_df.groupby(['patientid', 'time_day', 'variable_name'])['value'].agg(['min', 'max', 'mean']).reset_index()

# Pivot to wide format
vitals_labs_pivot = vitals_labs_agg.pivot_table(
    index=['patientid', 'time_day'],
    columns='variable_name',
    values=['min', 'max', 'mean']
)

# Flatten column names
vitals_labs_pivot.columns = [f'{var}_{stat}' for stat, var in vitals_labs_pivot.columns]
vitals_labs_pivot = vitals_labs_pivot.reset_index()

print(f"\n✓ Aggregated vitals/labs by day:")
print(f"  Shape: {vitals_labs_pivot.shape}")
print(f"  Features: {len([c for c in vitals_labs_pivot.columns if c not in ['patientid', 'time_day']])}")
print(f"  Example features: {list(vitals_labs_pivot.columns[2:8])}")

## PREPROCESS

In [8]:
# Calculate baseline P/F ratio (first 24h minimum - lower P/F is worse)
pf_df['charttime'] = pd.to_datetime(pf_df['charttime'])
pf_df['admission_time'] = pd.to_datetime(pf_df['admission_time'])

first_24h = pf_df[pf_df['charttime'] <= pf_df['admission_time'] + pd.Timedelta(hours=24)]

baseline_pf = first_24h.groupby('patientid')['pf_ratio'].min().reset_index()
baseline_pf.columns = ['patientid', 'baseline_pf_ratio']

print(f"\nBaseline P/F ratio:")
print(f"   Patients with baseline: {len(baseline_pf):,}")
print(f"   Mean: {baseline_pf['baseline_pf_ratio'].mean():.1f}")
print(f"   Median: {baseline_pf['baseline_pf_ratio'].median():.1f}")


Baseline P/F ratio:
   Patients with baseline: 4,566
   Mean: 187.4
   Median: 166.7


In [9]:
# Filter patients with sufficient measurements and baseline P/F < 300 (impaired oxygenation)
pf_counts = pf_df.groupby('patientid').size()
valid_patients = pf_counts[pf_counts >= 10].index

baseline_impaired = baseline_pf[baseline_pf['baseline_pf_ratio'] < 300]['patientid']
valid_patients = valid_patients.intersection(baseline_impaired)

pf_filtered = pf_df[pf_df['patientid'].isin(valid_patients)]
patient_final = patient_df[patient_df['patientid'].isin(valid_patients)].merge(
    baseline_pf, on='patientid', how='inner'
)

print(f"\nFiltering:")
print(f"   Patients with ≥10 P/F measurements: {len(patient_final):,}")
print(f"   Mean baseline P/F ratio: {patient_final['baseline_pf_ratio'].mean():.1f}")


Filtering:
   Patients with ≥10 P/F measurements: 2,450
   Mean baseline P/F ratio: 143.8


In [10]:
# Create time series
pf_ts = pf_filtered.merge(
    patient_final[['patientid', 'baseline_pf_ratio', 'admission_time']], 
    on=['patientid', 'admission_time'], 
    how='left'
)

pf_ts['time_days'] = (pf_ts['charttime'] - pf_ts['admission_time']).dt.total_seconds() / 86400
pf_ts['time_day'] = pf_ts['time_days'].astype(int)

# Rename for trajectory script
pf_ts = pf_ts.rename(columns={'pf_ratio': 'lab_value'})
pf_ts = pf_ts.drop_duplicates(subset=['patientid', 'charttime'])

print(f"\n✓ Time series created")
print(f"   Total measurements: {len(pf_ts):,}")
print(f"   Time range: {pf_ts['time_days'].min():.1f} to {pf_ts['time_days'].max():.1f} days")


✓ Time series created
   Total measurements: 56,042
   Time range: -0.0 to 28.0 days


In [11]:
# Save P/F ratio time series for trajectory computation
output_path = '../../results/hirid/ventilator/pf_ratio_timeseries.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

pf_ts.to_csv(output_path, index=False)

print(f"\n✓ Saved P/F ratio time series: {output_path}")
print(f"   Shape: {pf_ts.shape}")


✓ Saved P/F ratio time series: ../../results/hirid/ventilator/pf_ratio_timeseries.csv
   Shape: (56042, 9)


## TRAJECTORY MODELING

Run standalone script for SLURM:
```bash
python ../hirid_ventilator_trajs.py --window-days 3.0 --n-batches 10
```

In [ ]:
# Load pre-computed trajectory probabilities
trajectory_probs_path = '../../results/hirid/ventilator/ventilator_trajectory_probs_bayes.csv'

if os.path.exists(trajectory_probs_path):
    trajectory_probs = pd.read_csv(trajectory_probs_path)
    print(f"✓ Loaded trajectory probabilities from: {trajectory_probs_path}")
    print(f"   Shape: {trajectory_probs.shape}")
else:
    print(f"⚠️ File not found: {trajectory_probs_path}")
    print("   Run: python ../hirid_ventilator_trajs.py")

In [ ]:
# Visualize trajectory distribution
plot_trajectory_distribution(trajectory_probs)

## BIOMARKER SUMMARY STATISTICS

In [ ]:
LOOKBACK_DAYS = 3  # Match trajectory window for Ventilator (3 days)
pf_summary_3d = biomarker_summary_stats(pf_ts, 'lab_value', lookback_days=LOOKBACK_DAYS)
print(f"\n✓ Calculated P/F ratio summary stats over last {LOOKBACK_DAYS} days")
print(f"   Shape: {pf_summary_3d.shape}")
print(f"   Example stats:\n{pf_summary_3d.head()}")

# Save
pf_summary_3d.to_csv('../../results/hirid/ventilator/pf_ratio_summary_3d.csv', index=False)
print(f"   Saved: pf_ratio_summary_3d.csv")

## DEFINE OUTCOME

Successful ventilator weaning defined as:
1. P/F ratio improved to ≥ 400 (good oxygenation) AND
2. Sustained for 48 hours AND
3. Stable respiratory status (SpO2 ≥ 90%, RR < 30)

In [ ]:
# Configuration
PREDICTION_GAP_DAYS = 1.0  # 24-hour gap
PREDICTION_WINDOW_DAYS = 3.0  # Predict successful weaning within 3 days

print(f"📋 Prediction Configuration:")
print(f"   Gap:    {PREDICTION_GAP_DAYS} days")
print(f"   Window: {PREDICTION_WINDOW_DAYS} days")
print(f"   Total:  {PREDICTION_GAP_DAYS + PREDICTION_WINDOW_DAYS} days lookahead")

# Prepare respiratory parameter timing
resp_df['charttime'] = pd.to_datetime(resp_df['charttime'])
resp_df['admission_time'] = pd.to_datetime(resp_df['admission_time'])
resp_df['time_days'] = (resp_df['charttime'] - resp_df['admission_time']).dt.total_seconds() / 86400
resp_df['time_day'] = resp_df['time_days'].astype(int)

# Separate SpO2 and RR
spo2_df = resp_df[resp_df['variableid'].isin(['4000', '8280'])].copy()
rr_df = resp_df[resp_df['variableid'].isin(['300', '310', '5685'])].copy()

print(f"\n✓ Respiratory data prepared")
print(f"  SpO2 measurements: {len(spo2_df):,}")
print(f"  RR measurements: {len(rr_df):,}")

In [ ]:
# Define successful weaning events
weaning_events = []
excluded_counts = {'already_weaned': 0, 'no_future_data': 0}

for patientid, grp in patient_final.groupby('patientid'):
    grp = grp.sort_values('time_day')
    baseline_pf = grp.iloc[0]['pf_ratio']
    
    for i in range(len(grp)):
        current_time = grp.iloc[i]['time_day']
        current_pf_ratio = grp.iloc[i]['pf_ratio']
        
        # Skip if already has good oxygenation (P/F ratio ≥ 400)
        if current_pf_ratio >= 400.0:
            excluded_counts['already_weaned'] += 1
            continue
        
        # Define prediction window
        prediction_start = current_time + PREDICTION_GAP_DAYS + 1
        prediction_end = current_time + PREDICTION_GAP_DAYS + PREDICTION_WINDOW_DAYS + 1
        
        future_window = pf_ts[
            (pf_ts['time_days'].between(prediction_start, prediction_end, inclusive='both')) &
            (pf_ts['patientid'] == patientid)
        ]
        
        if len(future_window) == 0:
            excluded_counts['no_future_data'] += 1
            continue
        
        # Check for successful weaning: P/F ratio ≥ 400 sustained for 48h
        good_pf = future_window[future_window['pf_ratio'] >= 400.0]
        
        if len(good_pf) > 0:
            first_good_day = good_pf.iloc[0]['time_day']
            # Check if remained ≥ 400 for next 2 days
            sustained_window = pf_ts[
                (pf_ts['patientid'] == patientid) &
                (pf_ts['time_days'].between(first_good_day, first_good_day + 2, inclusive='both'))
            ]
            
            # Check respiratory stability
            spo2_window = spo2_df[
                (spo2_df['patientid'] == patientid) &
                (spo2_df['time_days'].between(first_good_day, first_good_day + 2, inclusive='both'))
            ]
            rr_window = rr_df[
                (rr_df['patientid'] == patientid) &
                (rr_df['time_days'].between(first_good_day, first_good_day + 2, inclusive='both'))
            ]
            
            pf_sustained = (sustained_window['pf_ratio'] >= 400.0).all() if len(sustained_window) > 0 else False
            spo2_stable = (spo2_window['value'] >= 90).mean() > 0.8 if len(spo2_window) > 0 else True
            rr_stable = (rr_window['value'] < 30).mean() > 0.8 if len(rr_window) > 0 else True
            
            target_weaning = int(pf_sustained and spo2_stable and rr_stable)
        else:
            target_weaning = 0
        
        weaning_events.append({
            'patientid': patientid,
            'time_day': current_time,
            'target_successful_weaning': target_weaning,
            'current_pf_ratio': current_pf_ratio,
            'baseline_pf_ratio': baseline_pf,
            'pf_improvement': current_pf_ratio - baseline_pf,
            'prediction_gap_days': PREDICTION_GAP_DAYS,
            'prediction_window_days': PREDICTION_WINDOW_DAYS
        })

weaning_outcomes_df = pd.DataFrame(weaning_events)

print(f"\n📊 Outcome Definition Summary:")
print(f"   Total prediction windows: {len(weaning_outcomes_df):,}")
print(f"   Unique patients: {weaning_outcomes_df['patientid'].nunique():,}")
print(f"   \n   Successful Weaning Events:")
print(f"      Total events: {weaning_outcomes_df['target_successful_weaning'].sum():,}")
print(f"      Event rate: {100*weaning_outcomes_df['target_successful_weaning'].mean():.1f}%")

## CONSTRUCT PREDICTION DATASET

In [ ]:
# Merge trajectory probabilities with outcomes
prediction_dataset = weaning_outcomes_df.copy()

if 'trajectory_probs' in locals() and 'prob_stable' in trajectory_probs.columns:
    prediction_dataset = prediction_dataset.merge(
        trajectory_probs[[
            'patientid', 'time_day', 
            'prob_stable', 'prob_gradual_decline', 'prob_rapid_decline'
        ]],
        on=['patientid', 'time_day'],
        how='left'
    )
    
    prediction_dataset['prob_declining'] = (
        prediction_dataset['prob_gradual_decline'] + 
        prediction_dataset['prob_rapid_decline']
    )

# Merge patient demographics
patient_static = patient_final[['patientid', 'age', 'sex']].drop_duplicates()
prediction_dataset = prediction_dataset.merge(
    patient_static,
    on='patientid',
    how='left'
)

# Merge P/F ratio summary statistics
prediction_dataset = prediction_dataset.merge(
    pf_summary_3d,
    on=['patientid', 'time_day'],
    how='left'
)

# Merge vitals/labs features
prediction_dataset = prediction_dataset.merge(
    vitals_labs_pivot,
    on=['patientid', 'time_day'],
    how='left'
)

print(f"\n📋 Final Prediction Dataset:")
print(f"   Rows: {len(prediction_dataset):,}")
print(f"   Columns: {len(prediction_dataset.columns)}")
print(f"   Unique patients: {prediction_dataset['patientid'].nunique():,}")

# Save
os.makedirs('../../results/hirid/ventilator', exist_ok=True)
prediction_dataset.to_csv('../../results/hirid/ventilator/ventilator_prediction_dataset.csv', index=False)
print(f"\n✓ Saved: results/hirid/ventilator/ventilator_prediction_dataset.csv")

## FIT MODELS

In [ ]:
# Extract dynamic vitals/labs features (those with _min, _max, _mean)
dynamic_features = [col for col in prediction_dataset.columns 
                     if any(s in col for s in ['_min', '_max', '_mean'])]

# Define feature sets (11 combinations matching MIMIC pipeline)
feature_sets = {
    'Trajectory Only': [
        'prob_stable',
        'prob_gradual_decline',
        'prob_rapid_decline',
        'prob_declining'
    ],
    
    'Summary Stats Only': [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']],
    
    'Trajectory + Summary Stats': [
        'prob_stable',
        'prob_gradual_decline',
        'prob_rapid_decline',
        'prob_declining'
    ] + [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']],
    
    'Static Only': [
        'pf_improvement',
        'baseline_pf_ratio',
        'age',
    ],
    
    'Trajectory + Static': [
        'prob_stable',
        'prob_gradual_decline',
        'prob_rapid_decline',
        'prob_declining',
        'baseline_pf_ratio',
        'pf_improvement',
        'age',
    ],
    
    'Summary Stats + Static': [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']] + [
        'pf_improvement',
        'baseline_pf_ratio',
        'age',
    ],
    
    'Trajectory + Summary Stats + Static': [
        'prob_stable',
        'prob_gradual_decline',
        'prob_rapid_decline',
        'prob_declining'
    ] + [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']] + [
        'pf_improvement',
        'baseline_pf_ratio',
        'age',
    ],
    
    'Static + Dynamic': [
        'pf_improvement',
        'baseline_pf_ratio',
        'age',
    ] + dynamic_features,
    
    'Trajectory + Static + Dynamic': [
        'prob_stable',
        'prob_gradual_decline',
        'prob_rapid_decline',
        'prob_declining',
        'baseline_pf_ratio',
        'pf_improvement',
        'age',
    ] + dynamic_features,
    
    'Summary Stats + Static + Dynamic': [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']] + [
        'pf_improvement',
        'baseline_pf_ratio',
        'age',
    ] + dynamic_features,
    
    'Trajectory + Summary Stats + Static + Dynamic': [
        'prob_stable',
        'prob_gradual_decline',
        'prob_rapid_decline',
        'prob_declining'
    ] + [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']] + [
        'pf_improvement',
        'baseline_pf_ratio',
        'age',
    ] + dynamic_features,
}

print(f"📋 Feature Set Summary (11 combinations):")
for name, features in feature_sets.items():
    print(f"   {name:45s}: {len(features):3d} features")

In [ ]:
# Prepare data for modeling
all_feature_cols = list(set([feat for features in feature_sets.values() for feat in features]))
prediction_clean = prediction_dataset.dropna(subset=['target_successful_weaning']).copy()

if 'sex' in prediction_clean.columns:
    prediction_clean['sex'] = prediction_clean['sex'].map({'m': 1, 'f': 0, 'M': 1, 'F': 0})

# Intelligent imputation strategy
# Trajectory probabilities: forward fill within patient with limit (recent trajectory state persists)
traj_cols = ['prob_stable', 'prob_gradual_decline', 'prob_rapid_decline', 'prob_declining']
for col in traj_cols:
    if col in prediction_clean.columns:
        prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=2)
        prediction_clean[col] = prediction_clean[col].fillna(0)  # Unknown states → 0

# Summary stats: forward fill with limit
summary_cols = [col for col in pf_summary_3d.columns if col not in ['patientid', 'time_day']]
for col in summary_cols:
    if col in prediction_clean.columns:
        prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=2)

# Vitals/labs: forward fill with longer limit (carry forward recent measurements)
for col in dynamic_features:
    if col in prediction_clean.columns:
        prediction_clean[col] = prediction_clean.groupby('patientid')[col].fillna(method='ffill', limit=3)

# Use shared impute_features for remaining missing values
prediction_clean = impute_features(prediction_clean, all_feature_cols)

y = prediction_clean['target_successful_weaning']
groups = prediction_clean['patientid']

print(f"\n📋 Final Dataset for Modeling:")
print(f"   Total samples: {len(y):,}")
print(f"   Positive class: {y.sum():,} ({100*y.mean():.1f}%)")
print(f"   Unique patients: {groups.nunique():,}")
print(f"\nImputation strategy:")
print(f"   Trajectories: ffill limit=2, then 0 (recent state persists)")
print(f"   Summary stats: ffill limit=2 (avoid stale aggregates)")
print(f"   Vitals/Labs: ffill limit=3 (carry forward measurements)")
print(f"   Remaining: median imputation")

## EVALUATE

In [ ]:
# Repeated Cross-validation
n_repeats = 10
n_folds = 5
n_folds_total = n_repeats * n_folds

comparison_results = {}

print(f"\n🎯 Training models with {n_repeats}-Repeat {n_folds}-Fold CV:\n")

for feature_set_name, feature_cols in feature_sets.items():
    print(f"{feature_set_name}")
    print("-" * 60)
    
    fold_metrics = train_repeated_cv(
        prediction_df=prediction_clean,
        feature_cols=feature_cols,
        target_col='target_successful_weaning',
        group_col='patientid',
        n_repeats=n_repeats,
        n_folds=n_folds,
        random_state=920
    )
    
    comparison_results[feature_set_name] = fold_metrics
    
    print(f"  ROC-AUC:           {np.mean(fold_metrics['roc_auc']):.3f} ± {np.std(fold_metrics['roc_auc']):.3f}")
    print(f"  Average Precision: {np.mean(fold_metrics['avg_precision']):.3f} ± {np.std(fold_metrics['avg_precision']):.3f}\n")

In [ ]:
# Performance comparison
summary_df = pd.DataFrame([
    {
        'Model': name,
        'ROC-AUC': f"{np.mean(metrics['roc_auc']):.3f} ± {np.std(metrics['roc_auc']):.3f}",
        'Avg Precision': f"{np.mean(metrics['avg_precision']):.3f} ± {np.std(metrics['avg_precision']):.3f}"
    }
    for name, metrics in comparison_results.items()
])

print("\n📊 Performance Summary:")
print(summary_df.to_string(index=False))

# ROC & PR Curves
fig, (ax1, ax2) = plot_roc_pr_curves(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_successful_weaning',
    color_scheme='Set1'
)
plt.savefig('../../results/hirid/ventilator/model_comparison_curves.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/ventilator/model_comparison_curves.png")
plt.show()

In [ ]:
# Statistical significance testing
pairs_to_compare = [(0, 1), (1, 2)]

fig, (ax1, ax2) = plot_boxplots_with_stats(
    comparison_results=comparison_results,
    outcome_df=prediction_clean,
    target_col='target_successful_weaning',
    pairs_to_compare=pairs_to_compare,
    n_folds_total=n_folds_total
)
plt.savefig('../../results/hirid/ventilator/model_comparison_boxplots.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: results/hirid/ventilator/model_comparison_boxplots.png")
plt.show()

print_statistical_comparisons(
    comparison_results=comparison_results,
    pairs_to_compare=pairs_to_compare
)